# 2nd Deep Learning Task - LLM fine tuning
By Dominykas Venclovas, LSP nr. - 2425052.

For this task Qwen2.5-1.5B model has been used for his relative small size and strong multilingual capabilities. LLM will be fine tuned using insurance rules based data - Q&A format, more on that later. Now the original model is loaded and tested.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")

def ask(question):
    messages = [
        {"role": "system", "content": "Tu esi draudimo ekspertas. Atsakyk trumpai ir tiksliai lietuvių kalba."},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

questions = [
    "Kas yra draudimo išskaita?",
    "Kokiais atvejais nemokama būsto draudimo išmoka?",
    "Ar draudžiami būste esantys grynieji pinigai?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 10)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Q: Kas yra draudimo išskaita?
A: Draudimo išskaita yra svarbi elementas, kuris jau buvo įtrauktas nuo Antikytri. Štai keli paieščioje:

1. Draudimo šaltinis: Jis buvo įtrauktas 3000 metų Antikytri.

2. Vieniniai: Taip pat buvo įtraukti vieniniai.

3. Sąjungos: Tęsėjo daugiau nei 5000 skirtingus vandeninių sūnų.

4. Draižiai: Pavyzdžiui, švietimas su pirmajame dairiu (Venus).

5. Draižininkai: Yra didesnių skaičių draitinių žmonių antokravimų.


----------
Q: Kokiais atvejais nemokama būsto draudimo išmoka?
A: Būsto draudimo išmoka yra naudojamas bei turėtų tokius pavyzdžius:

1. Šalys, kurie reikia užbūti ar šilumą

2. Pirmoji darbas - kiekvienas dalykas

3. Tarybos darbai - kaip įvertinimas, papildomos informacijos

4. Komentarai / reakcijos

5. Būdai, kad pasiruošite su kliente / pristatymui

6. Privalumas apimačių galimybes

7. Kategorijos (pavyzdžiui: kokybiška, televizija)

8. Dabar istorijos

9. Vardavimo procesas

10. Sutarties sąlygas


----------
Q: Ar draudžiami būste esantys

### Dataset preparation
For Lithuanian dataset preparation chunks of coherent text/table images from an insurance firm rules (many long .pdf files of such rules exist) are fed to LLM chat-bot with a goal of preparing question answer pairs. These pairs are later hand checked for truth and coherence, with inadequtes being removed or corrected.

Link for such rules available here: https://www.ld.lt/draudimo-taisykles

Use for such dataset is quite obvious - it could help answer questions about insurance, for clients or junior insurance consultants. Each countries and insurance companies policy rules may be different and wrapped in complicated legal language, therefore it could be hard for a lay person to understand or take a long time, even if they knew where to look. And chances that a general LLM with correctly answer such questions without finetuninng or context are quite slim.

Possible LLM chatbot queries:

* Given the following Lithuanian insurance rules text, generate 5-10 question and answer pairs in Lithuanian. Questions should reflect what a person might realistically ask. Answers should be accurate and grounded only in the provided image of text.
  
  Respond in JSON format: [{"question": "...", "answer": "..."}, ...]
  
  Given the following Lithuanian insurance rules text, generate 3-5
  question and answer pairs in Lithuanian. Questions should reflect
  what a person might realistically ask. Answers should be accurate
  and grounded only in the provided text.

  Text: {chunk}

  Respond in JSON format:
  [{"question": "...", "answer": "..."}, ...]
* Given the following Lithuanian insurance rules text, generate 5-10
question and answer pairs in Lithuanian. Questions should reflect what a person might realistically ask. Answers should be accurate and grounded only in the provided image of text.

  Respond in JSON format: [{"question": "...", "answer": "..."}, ...]

In [1]:
q_a_pairs = [
  {
    "question": "Kaip nustatoma maksimali draudimo suma?",
    "answer": "Draudimo suma nustatoma draudiko ir draudėjo susitarimu (apskaičiuojant pagal turto atkūrimo arba likutinę vertę), tačiau ji jokiu būdu negali viršyti draudžiamo turto vertės."
  },
  {
    "question": "Ar apdraustas mano dviratis, jei jį pavogs gatvėje, o ne iš namų?",
    "answer": "Taip, dviračių (taip pat elektrinių), paspirtukų, riedžių, vaikų ir neįgaliojo vežimėlių vagystė už draudimo vietos ribų yra įtraukta į visus namų turto draudimo variantus (Minimalų, Standartinį ir Maksimalų)."
  },
  {
    "question": "Netyčia užliejau žemiau gyvenančius kaimynus. Ar draudimas padengs jiems padarytą žalą?",
    "answer": "Taip, apdraustame bute arba name kilęs įvykis, padaręs žalą tretiesiems asmenims ar jų turtui (pvz., kaimynų užliejimas), yra atlyginamas pagal Civilinės atsakomybės draudimą."
  },
  {
    "question": "Jei dėl gaisro ar audros laikinai negalėsiu gyventi savo namuose, ar apmokėsite nuomos išlaidas kitur?",
    "answer": "Taip, gyvenamosios vietos nuomos išlaidos, laikinai praradus būstą, yra padengiamos, tačiau tam reikia turėti „Standartinį“ arba „Maksimalų“ pastatų draudimo variantą."
  },
  {
    "question": "Ar gausiu draudimo išmoką, jei iš užrakinto automobilio bus pavogti mano asmeniniai daiktai?",
    "answer": "Taip, vagystė su įsilaužimu iš užrakinto automobilio yra atlyginama, jeigu esate pasirinkę „Standartinį“ arba „Maksimalų“ namų turto draudimo variantą."
  },
  {
    "question": "Mano šuo apgadino kito žmogaus turtą. Ar draudimas tai kompensuos?",
    "answer": "Taip, Asmens civilinės atsakomybės draudimas apima gyvūnų veiksmus, dėl kurių kilo žala tretiesiems asmenims arba jų turtui."
  },
  {
    "question": "Ar minimalus pastatų draudimas atlygins žalą, jei suduš lango stiklas?",
    "answer": "Taip, stiklo dūžis yra įtrauktas į „Minimalų“, „Standartinį“ ir „Maksimalų“ pastatų draudimo variantus."
  },
  {
    "question": "Dėl elektros įtampos svyravimų sudegė mano buitinė technika. Ar tai draudžiamasis įvykis?",
    "answer": "Taip, turto sugadinimas arba sunaikinimas dėl elektros įtampos svyravimų ir viršįtampio yra atlyginamas, jei turite „Standartinį“ arba „Maksimalų“ namų turto (arba pastatų) draudimo variantą."
  },
 {
    "question": "Mano šuo namuose apgraužė duris. Ar draudimas atlygins šiuos nuostolius?",
    "answer": "Ne, draudimo apsauga netaikoma nuostoliams, atsiradusiems dėl naminių gyvūnų padarytos žalos apdraustam turtui."
  },
  {
    "question": "Netyčia sudaužiau savo mobiliojo telefono ekraną. Ar tai bus kompensuojama pagal stiklo dūžio draudimą?",
    "answer": "Ne, stiklo dūžio draudimas neatlygina žalos, padarytos mobiliųjų telefonų, planšetinių kompiuterių ir jų ekranams."
  },
  {
    "question": "Apvogė mano namus, bet vagys įėjo pro pamirštas užrakinti duris, todėl įsilaužimo žymių nėra. Ar gausiu draudimo išmoką?",
    "answer": "Ne, draudimo apsauga nuo vagystės netaikoma, jei į patalpas patekta pro neužrakintas duris ar neuždarytą langą ir nėra įsilaužimo požymių."
  },
  {
    "question": "Savo bute turiu nedidelę kirpyklą, kurioje dirbu ir priimu klientus. Ar joje esantiems daiktams galioja namų turto draudimas?",
    "answer": "Ne, namų turto draudimas netaikomas turtui tose patalpose, kuriose vykdoma komercinė ar gamybinė veikla."
  },
  {
    "question": "Gyvenu prie upės, kuri patvinsta kas kelerius metus. Ar draudimas padengs nuostolius dėl tokio potvynio?",
    "answer": "Ne, gamtos jėgų draudimas netaikomas, jei nuostolių kyla dėl potvynio, kuris draudimo vietoje buvo numatomas (t. y. įvyksta bent 1 kartą per 10 metų)."
  },
  {
    "question": "Sugedo mano 12 metų senumo šildymo katilas. Ar atlyginsite jo remonto išlaidas?",
    "answer": "Ne, draudimas neatlygina gedimų šildymo sistemos įrenginiams, kurie yra senesni nei 10 metų."
  },
  {
    "question": "Lyginant drabužius lygintuvu netyčia pradeginau brangią staltiesę, bet gaisras neįsiplieskė. Ar gausiu kompensaciją?",
    "answer": "Ne, žala dėl šilumos poveikio lyginant neatlyginama, išskyrus atvejus, kai gaisras išplito ir sugadino kitą apdraustą turtą."
  },
  {
    "question": "Plaudamas grindis išliejau kibirą vandens ir sugadinau brangų kilimą. Ar tai draudžiamasis įvykis?",
    "answer": "Ne, apsauga nuo vandens padarytos žalos netaikoma, jei vanduo buvo naudotas plaunant ar valant."
  },
  {
    "question": "Planuoju daryti kapitalinį buto remontą. Ar draudimo apsauga nesikeis?",
    "answer": "Pradėjus kapitalinį remontą ar rekonstrukciją, draudimo apsauga galioja tik ugnies ir (arba) gamtinių jėgų rizikoms. Pabaigus darbus, būtina pakeisti sutartį, kad vėl įsigaliotų platesnė apsauga."
  },
  {
    "question": "Ar yra kokių nors ribojimų daiktams, kuriuos nešiojuosi su savimi už namų ribų?",
    "answer": "Taip, tam tikriems įvykiams (pavyzdžiui, apiplešimui už draudimo vietos ribų) bei specifiniams daiktams (vertybėms, mobiliajai technikai) yra taikomi išmokų apribojimai. Šie apribojimai yra nurodyti taisyklėse ir draudimo liudijime."
  },
  {
    "question": "Ar mano turtas yra saugus, kai su juo keliauju?",
    "answer": "Draudimo apsauga turtui, kai jis yra ne draudimo vietoje (pavyzdžiui, keliaujant), gali turėti tam tikrų apribojimų."
  },
  {
    "question": "Apdraudžiau dar nebaigtą statyti namą. Nuo ko jis bus apsaugotas?",
    "answer": "Apdraudus nebaigtą statyti pastatą, apsauga galioja tik ugnies ir (arba) gamtinių jėgų rizikoms, o konkretus galiojimas priklauso nuo pastato baigtumo lygio."
  },
  {
    "question": "Kur galioja mano asmens civilinės atsakomybės draudimas?",
    "answer": "Priklausomai nuo jūsų pasirinkimo, šis draudimas gali galioti Lietuvoje, ES ir kitose išvardintose šalyse arba visame pasaulyje, išskyrus JAV, Kanadą, Australiją, Rusijos Federaciją ir Baltarusijos Respubliką."
  },
  {
    "question": "Ką turiu daryti, jei pamamečiau namų raktus?",
    "answer": "Praradus raktus, privalote per 24 valandas informuoti „Lietuvos draudimą“ ir policiją. Nuostoliai, atsiradę nuo raktų praradimo iki informavimo momento, nebus atlyginami."
  },
  {
    "question": "Per kiek laiko turiu pranešti apie įvykį, jei apdraustame bute trūko vamzdis?",
    "answer": "Atsitikus draudžiamajam įvykiui, turite nedelsiant informuoti reikiamas institucijas ir per 3 darbo dienas pranešti „Lietuvos draudimui“."
  },
  {
    "question": "Ar galiu už draudimą mokėti dalimis?",
    "answer": "Taip, už draudimo sutartį galite mokėti dalimis (nuo 2 iki 12 kartų), jei pasirenkate automatinį mokėjimo būdą."
  },
  {
    "question": "Kaip galiu nutraukti draudimo sutartį?",
    "answer": "Sutartį galite nutraukti kreipdamiesi į jus aptarnaujantį atstovą, skambindami tel. 1828, užsukę į skyrių arba savarankiškai „Lietuvos draudimo“ mobiliojoje programėlėje bei savitarnoje www.savasld.lt."
  },
  {
    "question": "Kokia yra mano kaip draudėjo atsakomybė rūpinantis turtu?",
    "answer": "Jūs privalote rūpintis, kad stogo danga būtų tvarkinga, sniegas nuvalytas, o vamzdynai, elektros instaliacija, spynos ir signalizacija būtų nuolat prižiūrimi ir tvarkingi."
  },
  {
    "question": "Kada tiksliai įsigalioja mano būsto draudimas?",
    "answer": "Draudimo apsauga įsigalioja draudimo liudijime nurodytą dieną atlikus mokėjimą, nebent sutartyje numatyta kitaip."
  },
  {
    "question": "Ar namų turto draudimas saugo mano daiktus, jei juos laikinai išsinešu iš namų?",
    "answer": "Taip, namų turto draudimas galioja ir kitose vietose, jei daiktus išsinešėte laikinai. Apsauga galioja Lietuvoje, Baltijos šalyse arba beveik visoje Europoje (išskyrus Rusiją ir Baltarusiją), priklausomai nuo pasirinkto varianto."
  },
  {
    "question": "Kas pagal būsto draudimo taisykles yra laikoma pastatais?",
    "answer": "Pastatai apima patį statinį ir jo nejudamas dalis (sienas, lubas, grindis, duris), taip pat šalia būsto esančius stacionarius statinius, pavyzdžiui, pagalbinius pastatus, tvoras ar garažo vartus."
  },
  {
    "question": "Ar mano baldai ir buitinė technika bus apdrausti?",
    "answer": "Taip, tai yra laikoma namų turtu. Namų turtas apima Jūsų turimus daiktus, pavyzdžiui, baldus, kilimus, vaizdo ir garso aparatūrą bei buitinę techniką."
  },
  {
    "question": "Kas nutiktų, jei dėl sugedusio vamzdžio užliečiau kaimynų butą?",
    "answer": "Tokiu atveju galiotų civilinės atsakomybės draudimas. Tai yra Jūsų atsakomybė, jei netyčia dėl savo valdomo turto trūkumų sugadintumėte kitų asmenų turtą, pavyzdžiui, užlietumėte kaimynų butą."
  },
  {
    "question": "Ar draudimas padengs žalą, jei važiuodamas dviračiu netyčia sužaločiau kitą žmogų?",
    "answer": "Taip, tai numatyta asmens civilinės atsakomybės draudime. Ši draudimo dalis apima Jūsų atsakomybę, jei netyčia pakenktumėte kito asmens sveikatai ar turtui (pvz., važiuojant dviračiu ar Jūsų augintiniui apgadinus svetimą turtą)."
  },
  {
    "question": "Mano kaimynas sako, kad ugnis iš mano namo persimetė į jo pastatą. Kuris draudimas už tai atsakingas?",
    "answer": "Už žalą kaimyno turtui, atsiradusią dėl gaisro iš Jūsų namo, yra atsakingas civilinės atsakomybės draudimas."
  },
  {
    "question": "Kaip man žinoti, ar mano konkretus patirtas įvykis yra draudžiamasis?",
    "answer": "Norėdami pasitikslinti, ar konkretus įvykis yra draudžiamasis, turite vadovautis būsto draudimo taisyklėmis ir savo draudimo liudijime nurodytu pasirinktu draudimo rizikų paketu."
  },
  {
    "question": "Kur galiu rasti tikslius draudžiamų objektų apibrėžimus?",
    "answer": "Tikslūs apibrėžimai pateikiami atitinkamose taisyklių dalyse: pastatų ir namų turto – A.II dalyje, civilinės atsakomybės – A.VIII dalyje, o asmens civilinės atsakomybės – A.IX dalyje."
  },
  {
    "question": "Kas gali tapti draudėju pagal šias taisykles?",
    "answer": "Draudėju gali būti turto savininkas, asmuo, įsigijęs turtą lizingu, arba asmuo, naudojantis turtą pagal nuomos ar panaudos sutartį. Taip pat juo gali būti savininko sutuoktinis, vaikai ar kiti asmenys, gavę savininko sutikimą apdrausti turtą."
  },
  {
    "question": "Kas yra laikoma šeimos nariais šiame draudime?",
    "answer": "Šeimos nariais laikomi sutuoktiniai arba kartu su draudėju nurodytu adresu gyvenantys ir bendrą ūkį turintys: nesusituokę asmenys, vaikai (įvaikiai) bei jų partneriai ir vaikai, tėvai, seneliai, broliai ir seserys."
  },
  {
    "question": "Kas yra naudos gavėjas?",
    "answer": "Tai asmuo, kuris yra nurodytas kaip naudos gavėjas draudimo liudijime, arba asmuo, kuris yra turto savininkas draudžiamojo įvykio dieną."
  },
  {
    "question": "Ar sutartyje gali dalyvauti kiti asmenys be draudiko ir draudėjo?",
    "answer": "Taip, be pagrindinių šalių sutartyje gali dalyvauti naudos gavėjai, šeimos nariai bei apdraustieji."
  },
  {
    "question": "Kas patenka į „susijusio asmens“ sąvoką?",
    "answer": "Susijęs asmuo apima naudos gavėją, šeimos narius, asmenis, faktiškai gyvenančius apdraustame pastate, bei asmenis, esančius pastate su draudėjo ar jo šeimos nario leidimu."
  },
  {
    "question": "Kuo skiriasi nuolat gyvenamas būstas nuo ne nuolat gyvenamo?",
    "answer": "Nuolat gyvenamas būstas yra toks, kuriame su šeimos nariais gyvenate ne mažiau kaip 9 mėnesius per metus. Ne nuolat gyvenamas būstas yra toks, kuriame būnate mažiau nei 9 mėnesius per metus arba naudojate jį trumpalaikės nuomos tikslams."
  },
  {
    "question": "Kas yra laikoma trumpalaike nuoma pagal šias taisykles?",
    "answer": "Trumpalaikė nuoma yra nuomos rūšis, kai nuomos sutartis sudaroma trumpesniam nei 2 mėnesių terminui."
  },
  {
    "question": "Ar galiu apdrausti savo lėšomis atliktą buto remontą, jei esu tik nuomininkas?",
    "answer": "Taip, gali būti apdraudžiamos investicijos į nuomojamą pastatą ar patalpas, jei jūs esate nuomininkas ir šios investicijos yra pagrįstos atliktų darbų ar įsigytų medžiagų sąskaitomis."
  },
  {
    "question": "Kokie kiemo objektai yra priskiriami stacionariems kiemo statiniams?",
    "answer": "Tai žemės sklypo aplinkos įrangos elementai, tokie kaip tvora, kiemo vartai, pavėsinė, lauko baseinas, kiemo aikštelė, trinkelės, taip pat stikliniai arba polikarbonatiniai šiltnamiai su įgilintais pamatais."
  },
  {
    "question": "Ar apdraudus pagrindinį namą, automatiškai apsaugomi ir kiti kieme esantys statiniai?",
    "answer": "Apdraudžiant gyvenamąjį pastatą, kartu apdraudžiami ir nedidelės vertės pagalbiniai pastatai bei stacionarūs kiemo statiniai iki sutartyje nurodyto limito. Didesnės vertės objektai į sutartį turi būti įtraukiami atskirai."
  },
  {
    "question": "Kas yra laikoma pagalbiniu pastatu?",
    "answer": "Tai tuo pačiu adresu kaip ir gyvenamasis namas esantis pastatas, skirtas ten gyvenančiųjų nuolatinėms reikmėms tenkinti. Butui priklausantis sandėliukas, garažas ar stovėjimo vieta taip pat laikomi pagalbiniais pastatais, jei jie yra tame pačiame pastate kaip ir butas."
  },
  {
    "question": "Ar galiu apdrausti namą, kuriame dar vyksta statybos ir jis nepripažintas tinkamu naudoti?",
    "answer": "Taip, taisyklėse numatyta galimybė apdrausti nebaigtą statyti pastatą – tai toks pastatas, kuris dėl neužbaigtų statybos darbų negali būti naudojamas pagal paskirtį arba dar nėra pripažintas tinkamu naudoti."
  },
  {
    "question": "Kas yra laikoma namų turtu šiame draudime?",
    "answer": "Namų turtas – tai namų apstatymo, buitinėms reikmėms skirti kilnojamieji daiktai, taip pat darbo įrankiai, baldai, buitinė technika, judėjimo priemonės (dviračiai, paspirtukai) ir kiti lauke pagal paskirtį laikomi stacionariai nepritvirtinti daiktai."
  },
  {
    "question": "Ar mano darbo įrankiai, kuriuos naudoju versle, yra apdrausti?",
    "answer": "Taip, darbo įrankiai ir įranga, naudojami profesinėje ar verslo veikloje, yra priskiriami namų turtui, tačiau jų draudimo suma yra ribojama iki 3 000 EUR."
  },
  {
    "question": "Ar draudimas galioja mano papuošalams ir gryniesiems pinigams?",
    "answer": "Grynieji pinigai ir papuošalai yra priskiriami vertybėms. Vertybės laikomos apdraustomis tik pasirinkus Standartinį arba Maksimalų draudimo variantą, o joms taikomi sutartyje numatyti išmokų limitai."
  },
  {
    "question": "Nuo kokios vertės meno kūriniai pradedami laikyti vertybėmis?",
    "answer": "Kailio ir odos dirbiniai bei meno kūriniai (paveikslai, skulptūros, kilimai) laikomi vertybėmis, kai jų vieno vertė yra didesnė nei 1 500 EUR."
  },
  {
    "question": "Ar yra specialus limitas mano mobiliajam telefonui ar laikrodžiui?",
    "answer": "Taip, laikrodžiams, mobiliesiems telefonams, muzikos instrumentams ir medicinos prietaisams taikomas vertybių limitas, kai jų vieno vertė viršija 2 000 EUR."
  },
  {
    "question": "Kas yra laikoma antikvariniais daiktais?",
    "answer": "Antkvariniai daiktai – tai visi prieš 50 metų ir anksčiau sukurti kilnojamieji materialūs visuomenės ir žmogaus veiklos kūriniai ar jų dalys, turintys išliekamąją vertę."
  },
  {
    "question": "Jei po gaisro mano butas taps netinkamas gyventi, ar draudimas padengs nuomos išlaidas?",
    "answer": "Taip, jei dėl draudžiamojo įvykio pastatas tampa netinkamas gyventi, gali būti apmokamos gyvenamosios vietos nuomos išlaidos arba atlyginamas nuomos pajamų netekimas."
  },
  {
    "question": "Ar apdraudžiami daiktai, esantys pagalbinėse patalpose (pvz., sandėliuke)?",
    "answer": "Taip, draudžiant namų turtą gyvenamajame pastate, kartu apdraudžiamas ir nedidelės vertės namų turtas, esantis pagalbinėse patalpose tuo pačiu adresu, neviršijant sutartyje nurodyto limito."
  },
  {
    "question": "Ar mano namuose auginami egzotiniai paukščiai ir žuvys yra apdrausti?",
    "answer": "Ne, pagal šias taisykles augalai, paukščiai, žuvys ir kiti gyvūnai yra laikomi nedraudžiamais objektais."
  },
  {
    "question": "Savo garaže laikau keturratį ir benzino atsargas. Ar šiems daiktams galioja draudimas?",
    "answer": "Ne, draudimas netaikomas visų rūšių transporto priemonėms, įskaitant keturračius, taip pat benzinui, dujoms ar dyzelinui."
  },
  {
    "question": "Ar draudimas atlygintų žalą, jei būtų sugadinta mano kompiuteryje esanti svarbi informacija ar programinė įranga?",
    "answer": "Draudimo apsauga netaikoma duomenų bazėms, informacijai laikmenose bei programinei įrangai, išskyrus tą įrangą, kuri buvo įdiegta perkant kompiuterį."
  },
  {
    "question": "Nusipirkau brangių statybinių medžiagų namo remontui. Ar jos yra apdraustos?",
    "answer": "Ne, statybinės medžiagos ir gaminiai yra priskiriami nedraudžiamiems objektams."
  },
  {
    "question": "Ar apdraustas mano kieme stovintis šiltnamis?",
    "answer": "Šiltnamiai nėra draudžiami, jei jie neturi stacionarių pamatų, jų karkasai pagaminti iš medžio ar plastiko arba jie yra dengti polietileno plėvele."
  },
  {
    "question": "Naudoju profesionalius įrankius savo verslui namuose. Ar jiems taikomi kokie nors ribojimai?",
    "answer": "Darbo įrankiai ir įranga, naudojami profesinėje ar verslo veikloje, nėra draudžiami, jei jų bendra vertė viršija 3 000 EUR sumą."
  },
  {
    "question": "Ar galiu apdrausti savo surinktą meno kūrinių kolekciją ar grynuosius pinigus?",
    "answer": "Šios vertybės yra nedraudžiamos, nebent turtą draudžiate Standartiniu arba Maksimaliu variantu arba dėl jų draudimo susitariama atskirai papildomose sutarties sąlygose."
  },
  {
    "question": "Kas nutinka, jei į draudžiamų objektų sąrašą per klaidą įtraukiamas nedraudžiamas daiktas?",
    "answer": "Tokiu atveju draudimo sutarties dalis dėl šių objektų negalioja, o draudikas, pastebėjęs tokią klaidą, perskaičiuoja įmoką ir grąžina permokėtą dalį klientui."
  },
  {
    "question": "Ar mano namų turtas yra apdraustas, jei jis laikomas kieme?",
    "answer": "Taip, namų turtas papildomai draudžiamas sklypo teritorijoje, jei Jūsų būstas yra nuolat gyvenamas, o turtas apdraustas pagal Standartinį arba Maksimalų variantą. Tačiau apsauga galioja tik tam turtui, kuris pagal savo paskirtį gali būti laikomas ne patalpoje."
  },
  {
    "question": "Kokia apsauga taikoma daiktams neaptverto sklypo teritorijoje?",
    "answer": "Neaptvertoje teritorijoje namų turtas yra saugomas tik nuo ugnies, gamtinių jėgų ir savaiminio medžio užvirtimo rizikų. Ši apsauga galioja tik pasirinkus Standartinį arba Maksimalų draudimo variantą."
  },
  {
    "question": "Ar mano papuošalai ir kiti vertingi daiktai yra apdrausti, jei juos palieku terasoje?",
    "answer": "Ne, draudimo apsauga vertybėms negalioja nei aptvertoje, nei neaptvertoje sklypo teritorijoje ar terasoje."
  },
  {
    "question": "Ar draudimas galioja daiktams, paliktiems terasoje, kuri nėra atskirta tvora?",
    "answer": "Apsauga aptverto sklypo teritorijoje (įskaitant terasą) negalioja, jeigu nėra stacionarių kliūčių patekti į tą teritoriją ar terasą."
  },
  {
    "question": "Kurioje šalyje turi būti pastatas, kad galiotų draudimas?",
    "answer": "Pastatai draudžiami tik tuo adresu, kuris nurodytas draudimo sutartyje ir yra Lietuvos Respublikos teritorijoje."
  },
  {
    "question": "Ar galiu tikėtis kompensacijos, jei neaptvertame kieme esantį turtą kas nors sugadino ar pavogė?",
    "answer": "Ne, neaptvertoje teritorijoje draudimo apsauga negalioja, jei turtas nukentėjo ne nuo ugnies, gamtinių jėgų arba savaiminio medžio užvirtimo."
  },
  {
    "question": "Ar mano asmeniniai daiktai yra apdrausti, kai esu ne namuose, pavyzdžiui, viešbutyje ar parduotuvėje?",
    "answer": "Taip, namų turto draudimas galioja laikinai jam esant už draudimo vietos ribų (pvz., parduotuvėje ar viešbutyje) ne ilgiau kaip 3 mėnesius nuo daikto atsiradimo kitoje vietoje datos. Apsauga vagystės ar vandalizmo atveju galioja tada, kai daiktai pavogti iš užrakintų patalpų arba nustatytas apiplėšimo faktas."
  },
  {
    "question": "Kokiose šalyse galioja laikinas daiktų draudimas?",
    "answer": "Galiojimo teritorija priklauso nuo pasirinkto varianto: Minimalus variantas galioja Lietuvos teritorijoje; Standartinis – Baltijos šalyse; Maksimalus – Europos Sąjungos valstybėse, Norvegijoje, Šveicarijoje, Vatikane, Islandijoje, San Marine, Andoroje, Monake, Lichtenšteine ir Jungtinėje Karalystėje."
  },
  {
    "question": "Ar draudimas galioja mano vaiko daiktams, jei jis laikinai išsikraustė studijuoti į kitą miestą?",
    "answer": "Draudimo apsauga negalioja, kai Jūs ar šeimos narys laikinai gyvenate kitur dėl studijų ar mokymosi, išskyrus tuos atvejus, kai sutartyje specialiai nurodytas studento daiktų draudimas."
  },
  {
    "question": "Planuoju persikraustyti į kitą būstą. Ką turiu padaryti, kad draudimas galiotų toliau?",
    "answer": "Apie persikraustymą privalote informuoti draudiką raštu iki persikraustymo pradžios, nurodydami naujo būsto adresą ir bendrą plotą."
  },
  {
    "question": "Kiek laiko galioja draudimas senajame būste po to, kai pranešu apie persikraustymą?",
    "answer": "Pranešus apie vietos keitimą, draudimo apsauga abiejose vietose (esančiose Lietuvoje) galioja ne ilgiau kaip 2 mėnesius nuo persikraustymo dienos."
  },
  {
    "question": "Ar draudimas atlygins žalą, jei mano daiktai bus sugadinti pačio pervežimo metu?",
    "answer": "Ne, nuostoliai, atsiradę pervežimo ir (arba) pernešimo metu, nėra atlyginami. Taip pat persikraustymo metu netaikomos laikino draudimo už draudimo vietos ribų nuostatos."
  },
  {
    "question": "Ar draudimas galioja brangiems papuošalams ar kitiems vertingiems daiktams, kai esu išvykęs iš namų?",
    "answer": "Draudimo apsauga už draudimo vietos ribų vertybėms negalioja, nebent sutartyje yra nurodyta kitaip pagal specifinius taisyklių punktus."
  },
{
    "question": "Ar gausiu išmoką, jei gaisras kilo dėl trumpojo jungimo elektros prietaise?",
    "answer": "Žala dėl trumpojo jungimo ar elektros įtampos svyravimų elektros prietaisuose nėra atlyginama, jei gaisras neišplito toliau."
  },
  {
    "question": "Ar draudimas galioja, jei kaminas buvo sugadintas jame degant suodžiams?",
    "answer": "Ne, žala dėl suodžių degimo kamine ir dėl to atsiradę nuostoliai pačiam kaminui ar jo konstrukcijai yra priskiriami nedraudžiamiesiems įvykiams."
  },
  {
    "question": "Ar bus atlyginta žala, jei į mano namą trenks žaibas?",
    "answer": "Taip, tiesioginis žaibo įtrenkimas į apdraustą turtą yra draudžiamasis įvykis. Taip pat atlyginami nuostoliai dėl žaibo užverstų medžių ar pastato dalių."
  },
  {
    "question": "Netyčia lygindamas drabužius pradeginau staltiesę. Ar tai draudžiamasis įvykis?",
    "answer": "Ne, žala dėl bet kokio šilumos poveikio lyginant, kepant, rūkant ar džiovinant nėra atlyginama, nebent gaisras išplistų ir sugadintų kitą apdraustą turtą."
  },
  {
    "question": "Ar draudimas padengs nuostolius, jei sprogs šildymo katilas?",
    "answer": "Sprogimas yra draudžiamasis įvykis, tačiau žala, padaryta tik pačiam kuro katilui dėl elektros tiekimo trikdžių ar nusidėvėjimo, nėra atlyginama (nors nuostoliai kitiems apdraustiems objektams tokio įvykio metu yra dengiami)."
  },
  {
    "question": "Kas nutiktų, jei į mano namą atsitrenktų nukritęs dronas?",
    "answer": "Valdomo skraidymo aparato, jo dalių ar krovinio užkritimas bei atsitrenkimas į objektus yra laikomas draudžiamuoju įvykiu."
  },
  {
    "question": "Ar dūmai ir suodžiai visais atvejais yra draudžiamasis įvykis?",
    "answer": "Dūmai, suodžiai ir žiežirbos yra draudžiamieji įvykiai, kai jie staiga išsiveržia iš gaisro vietos. Tačiau žala neatlyginama, jei dūmai ar suodžiai išsiskiria įprasto degimo metu (pvz., iš židinio ar žvakės) arba jei jie atsiranda dėl kondensato dūmtraukyje."
  },
  {
    "question": "Ar draudimas atlygins žalą, jei trūko vamzdis sienoje?",
    "answer": "Taip, žala dėl staigaus ir netikėto vandens išsiveržimo iš vamzdynų yra draudžiamasis įvykis. Nors už pačių paslėptų vamzdynų trūkimą neatlyginama, draudimas padengia nuostolius kitam turtui ir vamzdynų atidengimo bei sutvarkymo išlaidas."
  },
  {
    "question": "Kaimynai iš viršaus mane užliejo. Ar galiu kreiptis dėl kompensacijos?",
    "answer": "Taip, vandens prasiskverbimas iš gretimų, Jums nepriklausančių patalpų dėl ten įvykusios avarijos ar staigaus vandens išsiveržimo yra laikomas draudžiamuoju įvykiu."
  },
  {
    "question": "Po stiprios liūties vanduo pradėjo bėgti per stogą. Ar tai draudžiamas įvykis?",
    "answer": "Draudimas atlygina nuostolius už pirmą tokį įvykį, kai lietaus ar sniego tirpsmo vanduo staiga ir netikėtai prasiskverbia pro stogo dangą, sienas ar pamatus. Tačiau žala neatlyginama, jei įvykis kartojasi dažniau nei kas 4 metus."
  },
  {
    "question": "Pamiršau užsukti vandens čiaupą ir užliejau kambarį. Ar gausiu išmoką?",
    "answer": "Vandens išsiliejimas iš stacionarių įrenginių (pvz., palikus neužsuktą maišytuvą) yra draudžiamas, tačiau išmoka mokama tik už pirmą tokį įvykį per metus."
  },
  {
    "question": "Ar draudimas galioja, jei namas žiemą nebuvo šildomas ir jame trūko vamzdžiai?",
    "answer": "Ne, žala neatlyginama pastatams, kurie šildymo sezono metu nėra šildomi, jei juose yra įrengta šildymo sistema."
  },
  {
    "question": "Sprogo mano akvariumas ir sugadino parketą. Ar tai padengs draudimas?",
    "answer": "Taip, staigus ir netikėtas vandens išsiveržimas iš akvariumų, nesusijusių su vamzdžių sistema, yra įtrauktas į draudžiamųjų įvykių sąrašą."
  },
  {
    "question": "Ar draudimas apmoka už vandenį, kuris ištekėjo avarijos metu?",
    "answer": "Taip, padidėjusios išlaidos už išsiliejusį vandenį po draudžiamojo įvykio yra atlyginamos, jei jas galima pagrįsti komunalinių paslaugų sąskaitomis."
  },
  {
    "question": "Kokia liūtis yra laikoma draudžiamuoju įvykiu?",
    "answer": "Liūtis laikoma draudžiamuoju įvykiu, kai tai yra trumpalaikis intensyvus lietus, per kurį 12 valandų ar trumpesnį laiką iškrenta ne mažiau kaip 15 mm kritulių."
  },
  {
    "question": "Ar draudimas atlygins žalą, jei sniegas savo svoriu įlaužė stogą?",
    "answer": "Taip, sniego slėgis, kai sniegas savo svoriu sulaužo ar apgadina pastatus ir turtą, yra draudžiamasis įvykis. Tačiau žala neatlyginama, jei pastato konstrukcijos buvo nusidėvėjusios, nesandarios ar paveiktos puvinio."
  },
  {
    "question": "Mano namas stovi vietovėje, kurią dažnai semia. Ar potvynio žala bus kompensuojama?",
    "answer": "Žala dėl potvynio neatlyginama, jei jis draudimo vietoje yra numatomas, t. y. pagal statistinius duomenis pasikartoja bent 1 kartą per 10 metų."
  },
  {
    "question": "Vėjas išdaužė langą, nors didelės audros nebuvo. Ar tai draudžiamasis įvykis?",
    "answer": "Ne, žala langams ar durims, kurią padaro vėjas ne audros metu (pavyzdžiui, skersvėjis), yra priskiriama nedraudžiamiesiems įvykiams pagal šį punktą."
  },
  {
    "question": "Ar draudimas galioja, jei audros metu ant namo užvirto medis?",
    "answer": "Taip, atlyginami tiesioginiai nuostoliai, atsiradę per audrą ant apdrausto turto užvirtus medžiams ar jų šakoms."
  },
  {
    "question": "Kas yra laikoma nuošliauža pagal šias taisykles?",
    "answer": "Nuošliauža yra savaiminis grunto slinktis šlaitu žemyn, veikiant grunto svorio jėgai. Žala dėl šlaito erozijos, sukeltos paviršiumi tekančių vandenų, nėra atlyginama."
  },
  {
    "question": "Kaip nustatoma, ar vėjas ar liūtis atitiko draudimo sąlygas, jei arti nėra meteorologijos stoties?",
    "answer": "Tokiu atveju remiamasi faktais, ar tas pats įvykis padarė panašių nuostolių kitiems geros būklės pastatams ar daiktams draudimo vietos zonoje."
  },
  {
    "question": "Ar draudimas dengia žalą pastatams, kuriuose vyksta kapitalinis remontas?",
    "answer": "Žala pastatams, kuriuose vykdomi kapitalinio remonto ar rekonstrukcijos darbai, pagal šį skyrių neatlyginama, nebent tie elementai (stogas, langai, durys) nėra remontuojami ar keičiami."
  },
  {
    "question": "Jei vagys į namus pateko pro neuždarytą langą, ar draudimas atlygins nuostolius?",
    "answer": "Ne, draudimas neatlygina nuostolių, jei į patalpas pateko pro neuždarytą langą ar neužrakintas duris. Langas laikomas neuždarytu, jei jam uždaryti nepanaudoti (arba nevisiškai panaudoti) visi uždarymo mechanizmai."
  },
  {
    "question": "Ar mano daiktai saugūs, jei juos pavogė iš automobilio?",
    "answer": "Vagystė su įsilaužimu iš automobilio yra draudžiamasis įvykis, jei įveikiamos kliūtys (pvz., išlaužiama spyna ar išdaužiamas stiklas) ir jei turite pasirinkę Standartinį arba Maksimalų draudimo variantą. Tačiau žala pačiam automobiliui ar pavogtos vertybės tokiu atveju nėra atlyginamos."
  },
  {
    "question": "Kas yra laikoma apiplėšimu už draudimo vietos ribų?",
    "answer": "Tai atvejis, kai už apdrausto būsto ribų iš Jūsų ar Jūsų šeimos nario atimami su savimi turimi namų turto daiktai naudojant fizinę ar psichologinę prievartą. Kad galiotų ši apsauga, apie įvykį būtina pranešti policijai."
  },
  {
    "question": "Ar draudimas galioja daiktams, pavogtiems iš aptverto kiemo ar terasos?",
    "answer": "Taip, jei turite pasirinkę Standartinį arba Maksimalų variantą ir daiktai buvo pavogti iš visiškai aptvertos teritorijos, į kurią pašaliniai asmenys negalėjo patekti be kliūčių. Apsauga negalioja, jei į teritoriją pateksite pro atvirus vartus arba jei turtas pavogtas iš balkono."
  },
  {
    "question": "Ar apdraustas mano vaiko paspirtukas, jei jį pavogė iš mokyklos?",
    "answer": "Dviračių, paspirtukų ar vežimėlių vagystė už draudimo vietos ribų yra draudžiama, jei objektas pavagiamas iš užrakintų patalpų arba jei jis buvo prirakintas prie stacionaraus objekto. Įvykis būtinai turi būti užregistruotas policijoje."
  },
  {
    "question": "Ką daryti, jei pavogė mano namų raktus?",
    "answer": "Apie rakto vagystę turite nedelsiant, bet ne vėliau kaip per 24 valandas, pranešti policijai ir draudimo bendrovei. Tik tokiu atveju turtas, pagrobtas panaudojant šiuos raktus, bus laikomas draudžiamuoju įvykiu."
  },
  {
    "question": "Ar draudimas atlygins nuostolius dėl vandalizmo?",
    "answer": "Vandalizmas yra draudžiamasis įvykis, kai pastatas ar patalpa apgadinami mėginant įsilaužti arba jau įsilaužus į vidų. Jei pastatas sugadinamas iš išorės nesant įsilaužimo požymių, tokia žala laikoma nedraudžiamąja."
  },
  {
    "question": "Ar draudimas atlygins nuostolius, jei dėl elektros įtampos šuolio sugedo mano televizorius?",
    "answer": "Taip, turto sugadinimas dėl staigių ir netikėtų elektros įtampos svyravimų ar viršįtampių yra draudžiamasis įvykis. Tačiau tai turi būti vizualiai akivaizdu (pvz., perdegė saugikliai, atsirado apanglėjimų) arba patvirtinta remonto įmonės."
  },
  {
    "question": "Mano šaldytuvas tiesiog nustojo veikti dėl senatvės. Ar tai draudžiamasis įvykis?",
    "answer": "Ne, draudimas neatlygina nuostolių, jei gedimai atsirado dėl natūralaus susidėvėjimo, gamybos broko ar savaiminių gedimų."
  },
  {
    "question": "Ar apdrausti pastato išorėje esantys elektros įrenginiai?",
    "answer": "Jei draudžiami pastatai, atlyginami nuostoliai dėl pastato išorėje esančių elektros įrenginių (pvz., vėdinimo ar oro kondicionavimo sistemų) sugadinimo, be kurių sistema negali funkcionuoti."
  },
  {
    "question": "Sugedo mano vandens siurblio variklis. Ar galiu tikėtis kompensacijos?",
    "answer": "Elektros variklių gedimai yra draudžiami, tačiau apsauga negalioja, jei varikliui vis dar taikoma gamintojo garantija, jei jis yra senesnis nei 10 metų arba jei gedimas įvyko dėl natūralaus susidėvėjimo."
  },
  {
    "question": "Sprogo šildymo katilas ir vanduo užliejo grindis. Ar draudimas padengs išlaidas?",
    "answer": "Taip, draudimas atlygina išlaidas šildymo sistemos gedimams pašalinti bei dėl šių gedimų (pvz., išsiliejusio vandens) atsiradusią žalą pastatui ir namų turtui."
  },
  {
    "question": "Kada šildymo sistemos gedimas laikomas nedraudžiamuoju?",
    "answer": "Žala neatlyginama, jei įrenginiui galioja garantija, jei jis yra senesnis nei 10 metų (arba negalima nustatyti pagaminimo datos), taip pat savadarbiams ar nesertifikuotiems katilams."
  },
  {
    "question": "Ar draudimas galioja, jei prietaisas sugedo dėl netinkamo jo sumontavimo?",
    "answer": "Ne, nuostoliai neatlyginami, jei įrangos gedimai atsirado dėl netinkamo prijungimo, montavimo, netinkamos eksploatacijos ar netinkamai atliekamos techninės priežiūros."
  },
  {
    "question": "Kas yra apimama plačiausios apsaugos draudimu?",
    "answer": "Ši apsauga apima bet kokius staiga ir netikėtai įvykusius turto sugadinimo, sunaikinimo ar praradimo įvykius, išskyrus tuos, kurie nurodyti kaip nedraudžiamieji."
  },
  {
    "question": "Ar plačiausios apsaugos draudimas galioja mano išmaniajam telefonui?",
    "answer": "Mobiliesiems telefonams, planšetiniams ir nešiojamiesiems kompiuteriams ši apsauga galioja tik tuo atveju, jei tai yra nurodyta draudimo sutartyje kaip pasirinkta papildoma paslauga."
  },
  {
    "question": "Ar gausiu išmoką, jei netyčia subraižiau baldų paviršių?",
    "answer": "Ne, žala, kuri nedaro įtakos turto funkcionavimui, pavyzdžiui, estetiniai pakeitimai kaip subraižymai, įbrėžimai ar spalvos pakitimai, yra neatlyginama."
  },
  {
    "question": "Ar draudimas atlygins nuostolius, jei daiktą tiesiog pamečiau ar palikau?",
    "answer": "Ne, žala dėl turto pametimo, dingimo be įsilaužimo ar apiplėšimo požymių, taip pat dėl turto pasisavinimo ar sukčiavimo, nėra atlyginama."
  },
  {
    "question": "Kas nutiktų, jei nuomininkai tyčia sugadintų mano turtą?",
    "answer": "Nuostoliai dėl nuomininkų tyčia padarytos žalos atlyginami iki 3000 EUR sumos, jei yra sudaryta ilgalaikė nuomos sutartis (ilgesniam nei 2 mėn. laikotarpiui), o įvykis užregistruotas policijoje per 24 valandas."
  },
  {
    "question": "Ar draudimas galioja daiktams, naudojamiems sportui ar poilsiui, jei jie sulūžta?",
    "answer": "Žala dėl sportui ar poilsiui naudojamo daikto lūžimo ar susidėvėjimo neatlyginama, nebent sugadinimą lėmė kitas judantis asmuo ar daiktas."
  },
  {
    "question": "Ar atlyginama žala už sugedusią elektroniką dėl vidinių gedimų?",
    "answer": "Neatlyginama žala dėl mechaninio ar vidinio gedimo bei perdegimo, jei tai lėmė vidiniai veiksniai ar netinkamas eksploatavimas, o ne išorinis poveikis."
  },
  {
    "question": "Ar draudimas padengs banko palūkanas, jei mano įkeistas namas taps netinkamas gyventi po gaisro?",
    "answer": "Taip, draudimas atlygina palūkanas, mokamas už įkeistus apdraustus pastatus, kai jie dėl draudžiamojo įvykio tampa netinkami gyventi, tačiau šis atlyginimas mokamas ne ilgiau nei 3 mėnesius."
  },
  {
    "question": "Kokias išlaidas draudimas padengia sutvarkant vietą po avarijos?",
    "answer": "Atlyginamos išlaidos, skirtos pastato dalių nugriovimui, šiukšlių išvežimui, teritorijos sutvarkymui, vandens išsiurbimui, taip pat turto pervežimui bei sandėliavimui."
  },
  {
    "question": "Kiek laiko draudimas gali apmokėti mano daiktų sandėliavimą, jei negaliu jų laikyti namuose?",
    "answer": "Sandėliavimo išlaidos atlyginamos tol, kol pastatas vėl tampa tinkamas naudoti ir jame galima laikyti turtą, bet ne ilgiau nei 5 mėnesius."
  },
  {
    "question": "Ar draudimas apmokės spynų keitimą, jei pamėčiau raktus?",
    "answer": "Ne, išlaidos durų užrakto pakeitimui atlyginamos tik tada, jei raktai buvo pavogti ar atimti apiplėšimo būdu ir šis faktas yra užregistruotas policijoje."
  },
  {
    "question": "Ar mano vaiko daiktai yra apdrausti bendrabutyje ar nuomojamame bute studijų metu?",
    "answer": "Studento daiktų draudimas galioja kitoje gyvenamojoje vietoje Lietuvos Respublikos teritorijoje studijų metu (ne ilgiau kaip 5 metus), tačiau būtina turėti nuomos ar kitą sutartį, patvirtinančią gyvenamąją vietą."
  },
  {
    "question": "Ar studento daiktų draudimas galioja daiktams, paliktiems bendro naudojimo patalpose (pvz., koridoriuje ar virtuvėje)?",
    "answer": "Ne, draudimo apsauga negalioja, jei žala daiktams padaryta bendro naudojimo patalpose."
  },
  {
    "question": "Ar mano išmanusis telefonas yra apdraustas visais atvejais pagal plačiausią apsaugą?",
    "answer": "Mobiliųjų telefonų, planšetinių ir nešiojamųjų kompiuterių draudimas galioja tik papildomai pasirinkus šią paslaugą ir draudžiant turtą maksimaliu variantu. Apsauga netaikoma, jei per sutarties laikotarpį jau įvyko daugiau nei 2 tokie įvykiai."
  },
  {
    "question": "Kokia išmoka mokama, jei negaliu pateikti dokumentų, įrodančių telefono pirkimą?",
    "answer": "Nepateikus įsigijimo dokumentų ar kitų įrodymų apie daikto turėjimą, bei negalint tiksliai nustatyti jo senumo, atlyginama tik 30 % naujo prietaiso vertės."
  },
  {
    "question": "Kaip skaičiuojama išmoka už senesnį nešiojamąjį kompiuterį?",
    "answer": "Jei nešiojamasis kompiuteris įvykio dieną yra senesnis nei 4 metai (telefonas – senesnis nei 2 metai), nuostoliai atlyginami likutine verte, taikant nusidėvėjimą."
  },
  {
    "question": "Ar draudimas atlygina nuostolius už pas mane viešinčių svečių daiktus?",
    "answer": "Taip, atlyginami nuostoliai Jūsų svečių daiktams, esantiems draudimo vietoje, jei įvyksta draudžiamasis įvykis. Tačiau apsauga negalioja, jei tie daiktai jau yra apdrausti kita draudimo sutartimi."
  },
  {
    "question": "Kiek draudimas gali sumokėti už kito būsto nuomą, jei mano namai tapo netinkami gyventi?",
    "answer": "Pasirinkus Standartinį arba Maksimalų variantą, atlyginama iki 6 000 EUR už vieną įvykį, bet ne daugiau nei 500 EUR per mėnesį. Išmoka mokama ne ilgiau kaip 12 mėnesių."
  },
  {
    "question": "Kokių dokumentų reikia norint gauti kompensaciją už gyvenamosios vietos nuomą?",
    "answer": "Išlaidos turi būti pagrįstos paslaugų pirkimo dokumentais, pavyzdžiui, viešbučio sąskaitomis ar nuomos sutartimis su mokėjimo įrodymais."
  },
  {
    "question": "Kaip nustatomi civilinės atsakomybės draudimo limitai?",
    "answer": "Civilinė atsakomybė draudžiama iki draudimo liudijime nurodytų limitų, kurie priklauso nuo pasirinkto draudimo varianto. Jei draudžiami keli objektai, draudimo sumos nėra sumuojamos."
  },
  {
    "question": "Kieno kapaviečių paminklus galima apdrausti pagal šią sutartį?",
    "answer": "Atlyginami nuostoliai dėl žalos, padarytos Jūsų ar šeimos narių susijusių asmenų (tėvų, vaikų, senelių, brolių, seserų, tetų, dėdžių) kapavietėms. Šis sąrašas nėra baigtinis, todėl apsauga galioja ne tik artimų giminaičių kapavietėms."
  },
  {
    "question": "Kiek daugiausia kapaviečių gali būti apdrausta ir kokioje teritorijoje?",
    "answer": "Nuostoliai gali būti atlyginami daugiausia už 3 kapavietes, esančias bet kurioje Lietuvos Respublikos teritorijoje."
  },
  {
    "question": "Kokiems paminklams draudimo apsauga netaikoma?",
    "answer": "Draudimo apsauga negalioja laikiniesiems arba mediniams paminklams, taip pat paminklams, kurie neturi pamatų."
  },
  {
    "question": "Kokia draudimo apsauga galioja pastatams, kuriuose vyksta kapitalinis remontas ar rekonstrukcija?",
    "answer": "Jei dėl rizikų buvo susitarta sudarant sutartį, galioja apsauga tik nuo ugnies (kai keičiami išoriniai elementai) arba tik nuo ugnies ir gamtinių jėgų (kai išoriniai elementai nekeičiami). Visos kitos draudimo apsaugos pradeda galioti tik kitą dieną po darbų pabaigos."
  },
  {
    "question": "Kuo skiriasi paprastasis remontas nuo kapitalinio remonto?",
    "answer": "Paprastasis remontas – tai statinio atnaujinimas nekeičiant ir nestiprinant laikančiųjų konstrukcijų (pvz., dažomos sienos, keičiama grindų danga). Kapitalinis remontas – tai statinio pertvarkymas pakeičiant laikančiąsias konstrukcijas, bet nekeičiant statinio išorės matmenų."
  },
  {
    "question": "Kas laikoma pastato rekonstrukcija?",
    "answer": "Rekonstrukcija – tai statybos rūšis, kurios tikslas yra pertvarkyti statinį pakeičiant jo laikančiąsias konstrukcijas ir kartu pakeičiant bet kuriuos statinio išorės matmenis (ilgį, plotį, aukštį ir pan.)."
  },
  {
    "question": "Kuris draudimo variantas apima plačiausios apsaugos draudimą?",
    "answer": "Plačiausios apsaugos draudimas yra įtrauktas tik į Maksimalų pastatų draudimo variantą."
  },
  {
    "question": "Ar gausiu išmoką už vandens prasiskverbimą pro pastato konstrukcijas, jei pasirinkau Standartinį variantą?",
    "answer": "Taip, pasirinkus Standartinį variantą, už pirmąjį įvykį numatytas iki 1 500 EUR atlyginimas. Maksimalaus varianto atveju už pirmąjį įvykį šis limitas nėra taikomas."
  },
  {
    "question": "Kokie yra atlyginimo limitai už turto sugadinimą dėl elektros įtampos svyravimų?",
    "answer": "Standartinis variantas numato iki 7 000 EUR, o Maksimalus variantas – iki 16 000 EUR atlyginimą. Minimalus variantas šios rizikos neapdraudžia."
  },
  {
    "question": "Ar elektros variklių gedimai yra apdrausti visuose variantuose?",
    "answer": "Ne, elektros variklių gedimai yra apdrausti tik pasirinkus Maksimalų variantą, o numatyta išmokos riba yra iki 1 500 EUR."
  },
  {
    "question": "Kokia maksimali suma numatyta šildymo sistemos ir jos įrenginių gedimams?",
    "answer": "Šildymo sistemos gedimams Standartiniame variante numatyta iki 500 EUR, o Maksimaliame – iki 5 000 EUR suma."
  },
  {
    "question": "Kiek draudimas atlygins už papildomas išlaidas vietai sutvarkyti po įvykio?",
    "answer": "Atlyginama suma priklauso nuo pasirinkto varianto: Minimalus variantas dengia iki 2 %, Standartinis – iki 5 %, o Maksimalus – iki 7 % nuo draudimo sumos."
  },
  {
    "question": "Ar visuose variantuose apdraudžiami pagalbiniai pastatai ir kiemo statiniai?",
    "answer": "Taip, visuose trijuose variantuose (Minimaliame, Standartiniame ir Maksimaliame) pagalbiniams pastatams bei stacionariems kiemo statiniams taikomas iki 3 000 EUR limitas."
  },
  {
    "question": "Koks limitas taikomas gyvenamosios vietos nuomos išlaidoms?",
    "answer": "Standartiniame variante numatytas iki 6 000 EUR limitas. Maksimaliame variante taip pat numatyta iki 6 000 EUR suma, tačiau paminėta, kad už pirmąjį įvykį gali būti skiriama iki 1 000 EUR."
  },
  {
    "question": "Kas yra laikoma draudžiamuoju civilinės atsakomybės įvykiu?",
    "answer": "Tai su nekilnojamojo turto valdymu nesusiję Apdraustojo veiksmai (veikimas ar neveikimas), padarę žalos trečiojo asmens sveikatai, gyvybei ar turtui, kai reikalavimas atlyginti žalą pareiškiamas per draudimo laikotarpį arba per 30 dienų po jo pabaigos."
  },
  {
    "question": "Ar draudimas atlygins žalą, jei ją tyčia padarė mano vaikas?",
    "answer": "Taip, draudžiamuoju įvykiu laikoma žala, kilusi dėl vaikų iki 14 metų ir teisėtų globotinių veiksmų, net jei žala kilo dėl vaiko tyčinės veikos."
  },
  {
    "question": "Kokios taisyklės taikomos draudimui dėl naminių gyvūnų padarytos žalos?",
    "answer": "Draudimas atlygina žalą, padarytą Apdraustojo naminių gyvūnų, išskyrus arklius, galvijus ir kovinius ar pavojingų veislių šunis bei jų mišrūnus. Taip pat vienos rūšies suaugusių gyvūnų skaičius negali viršyti 5."
  },
  {
    "question": "Ar esu apdraustas, jei padarysiu žalą kitiems žaisdamas krepšinį ar dalyvaudamas neprofesionaliose varžybose?",
    "answer": "Taip, draudžiama Apdraustojo, kaip neprofesionalaus žaidėjo ar varžybų dalyvio, atsakomybė kitam tokio žaidimo ar varžybų dalyviui."
  },
  {
    "question": "Ar galioja draudimas naudojantis elektriniu paspirtuku ar dviračiu?",
    "answer": "Taip, draudžiama žala, padaryta valdant, naudojant ar kraunant nemotorines sausumos transporto priemones, pavyzdžiui, dviračius, riedžius ar elektrinius paspirtukus, kurių gamyklinis greitis neviršija 25 km/val., o galia – 1 kW."
  },
  {
    "question": "Kokie limitai taikomi žalai, padarytai išnuomotam kilnojamajam turtui?",
    "answer": "Draudimo išmokos limitas žalai, padarytai iš juridinių asmenų išsinuomotam kilnojamajam turtui, yra 3 000 EUR vienam įvykiui ir visam sutarties laikotarpiui (neatlyginamas nusidėvėjimas ar prekinės išvaizdos netekimas)."
  },
  {
    "question": "Ar draudimas galioja, jei žalą tretiesiems asmenims padaro mano namų darbininkas?",
    "answer": "Taip, draudžiama žala tretiesiems asmenims, padaryta Apdraustojo namų ūkyje įdarbinto darbuotojo veiksmais."
  },
  {
    "question": "Ar esu apdraustas, jei valdau droną?",
    "answer": "Draudžiama žala, padaryta valdant bepilotes skraidykles ne komerciniais tikslais, jei jų masė neviršija 900 g. Išmokos limitas yra 3 000 EUR vienam įvykiui ir visam laikotarpiui."
  }
]

len(q_a_pairs)

154

### Model fine tunning

In [2]:
!pip install -q --upgrade torchao peft trl transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 8.9 MB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import torch

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")

# Format into chat-style strings
def format_example(pair):
    messages = [
        {"role": "system", "content": "Tu esi draudimo ekspertas. Atsakyk trumpai ir tiksliai lietuvių kalba."},
        {"role": "user", "content": pair["question"]},
        {"role": "assistant", "content": pair["answer"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = Dataset.from_list([format_example(p) for p in q_a_pairs])

# LoRA config
lora_config = LoraConfig(
    r=8,  # adapter rank — higher = more capacity, more memory
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./qwen-insurance-lora",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        fp16=True,
    ),
)
trainer.train()

# Save the LoRA adapter only (small file, not the full model)
model.save_pretrained("./qwen-insurance-lora-final")
tokenizer.save_pretrained("./qwen-insurance-lora-final")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Adding EOS to train dataset:   0%|          | 0/154 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/154 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.676435
20,3.209238
30,2.918107
40,2.785734
50,2.568833
60,2.460317


('./qwen-insurance-lora-final/tokenizer_config.json',
 './qwen-insurance-lora-final/chat_template.jinja',
 './qwen-insurance-lora-final/tokenizer.json')

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "./qwen-insurance-lora-final"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)  # load from saved adapter folder
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, ADAPTER)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [6]:
def ask(question):
    messages = [
        {"role": "system", "content": "Tu esi draudimo ekspertas. Atsakyk trumpai ir tiksliai lietuvių kalba."},
        {"role": "user", "content": question}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

questions = [
    "Kas yra draudimo išskaita?",
    "Kokiais atvejais nemokama būsto draudimo išmoka?",
    "Ar draudžiami būste esantys grynieji pinigai?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("-" * 10)

Q: Kas yra draudimo išskaita?
A: Draudimo išskaita - štūros, gėlios ar kitų skiriamų medžiagų išskirtinės produktų apdorojimas.
----------
Q: Kokiais atvejais nemokama būsto draudimo išmoka?
A: Nemokama būsto draudimas yra stipriams priežastims, kad nepaiminkas darbas apima šaltinius ar kitos komercinis produktų, kurie nesuprantami buvo sukreiptos nevartojamas žmonėms.

Taip pat nemokamos būsto draudos išmokas:

1. Būstų draudos išmoks dėl neįtikimų (nepažeiskiami nuo draudimo)
2. Dėl nėra suteikta informacija apie produkto sutrikimus ar įtikimą
3. Dėl nepažymyti skirtingų dokumentų (pavyzdžiui, nėra skatinti visuomenėje informacijos apie kokybę, laiko ar teisinio pag
----------
Q: Ar draudžiami būste esantys grynieji pinigai?
A: Taip, buvo pasiektos štovėjus arba kryptis.
----------
